## DLA CODE OF CONDUCT V2.0

This Code of Conduct defines the principles governing ethical, transparent, and responsible use of Large Language Models (LLMs), online resources, and peer collaboration in the Deep Learning Applications laboratories. This version of the Code of Conduct was refined via a brainstorming session with **ChatGPT Version 5.2** and subsequently adapted to reflect the specific requirements and values of the DLA laboratories. In that spirit, this Code itself models the transparency it expects from you.

***Our goal is not to restrict innovation, but to ensure integrity, accountability, and genuine learning.***

### 1. Transparency in the Use of LLMs and AI Tools

The use of LLMs and AI-assisted tools is permitted — *but it must be transparent*.

* **Explicit Disclosure:** Clearly state if and how LLMs (e.g., ChatGPT, Copilot, Claude, etc.) were used. This includes code generation, debugging, data analysis, experiment design, report writing, or conceptual clarification.
* **Description of Contribution:** Briefly describe what the tool contributed and how you modified, verified, or extended its output.
* **Acknowledgment of Limitations:** Recognize that LLM outputs may contain errors, biases, or non-optimal solutions. You are responsible for verifying correctness, appropriateness, and academic integrity.

***Using AI does not reduce your responsibility for the final result.***

### 2. Proper Attribution and Documentation

Deep learning builds on existing work — responsibly.

* **Attribution:** Properly cite all external resources, including: Code snippets, Tutorials, Documentation, Datasets, Pretrained models, Research papers, and AI-generated content.
* **Reproducibility:** Clearly document tools, libraries, model versions, hyperparameters, and experimental setups so that your work can be reproduced.
* **Clarity of Modifications:** If you adapt external code, explicitly indicate what you changed and why.

***Transparency is a sign of scientific maturity — not weakness.***

### 3. Collaboration and Individual Responsibility

Discussion is encouraged. Copying is not.

* **Collaborative Learning:** You are encouraged to discuss concepts, debugging strategies, and approaches with classmates.
* **Individual Submission:** Your submitted solution must reflect your own understanding and implementation.
* **No Direct Sharing of Solutions:** Do not share complete solutions, trained models, or reports. Do not submit another person's work — or AI-generated work — as your own without meaningful engagement and proper disclosure.

***If you cannot explain your submission, it is not your submission.***

### 4. Accountability and Academic Integrity

You are responsible for everything you submit. Failure to comply with these guidelines may result in review by the course examination commission and can lead to disciplinary measures in accordance with university regulations.

***Integrity is part of your training as a machine learning practitioner.***

### 5. The Spirit of This Code of Conduct

This course prepares you to work in a field where:

* Reproducibility matters
* Ethical considerations matter
* Transparency matters
* Responsible AI use matters

***The purpose of this Code of Conduct is not surveillance — it is professional formation.***

### TL;DR

Use AI; Don’t let AI use you; Be transparent; Cite everything; Do your own thinking.

***If you can’t explain it, you probably shouldn’t submit it.***

---
---

## Introduction

In this second laboratory we will gain some experience working with Transformer models for a variety tasks using (mostly) the Hugging Face Ecosystem. 


---
### Exercise 1: Sentiment Analysis (warm up)

In this first exercise we will start from a pre-trained BERT transformer and build up a model able to perform text sentiment analysis. Transformers are complex beasts, so we will build up our pipeline in several explorative and incremental steps.

#### Exercise 1.1: Loading the Dataset Splits
There are a many sentiment analysis datasets, but we will use one of the smallest ones available: the [Cornell Rotten Tomatoes movie review dataset](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes), which consists of 5,331 positive and 5,331 negative processed sentences from the Rotten Tomatoes movie reviews.

**Your first task**: Load the dataset and figure out what splits are available and how to get them. Spend some time exploring the dataset to see how it is organized. Note that we will be using the [HuggingFace Datasets](https://huggingface.co/docs/datasets/en/index) library for downloading, accessing, splitting, and batching data for training and evaluation.

In [1]:

# Dataset imports.
from datasets import load_dataset, get_dataset_split_names

# load_dataset scarica il dataset, get_dataset_split_names ci permette di interrogare il dataset per scoprire la sua struttura (train/validation/test)
split = get_dataset_split_names("cornell-movie-review-data/rotten_tomatoes")

C:\Users\Utente\Desktop\DLA_HW\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
split

['train', 'validation', 'test']

proviamo già a scaricarli

In [3]:
# load all splits in un dict
ds_dict ={split: load_dataset("cornell-movie-review-data/rotten_tomatoes", split=split) for split in split}

In [4]:
ds_dict

{'train': Dataset({
     features: ['text', 'label'],
     num_rows: 8530
 }),
 'validation': Dataset({
     features: ['text', 'label'],
     num_rows: 1066
 }),
 'test': Dataset({
     features: ['text', 'label'],
     num_rows: 1066
 })}

In [5]:
import numpy as np

ds_train = ds_dict["train"]
for row in np.random.permutation(len(ds_train))[:18]:
    print(f"{ds_train[row]['label']}: {ds_train[row]['text']}")

0: an overstuffed compendium of teen-catholic-movie dogma .
1: " spider-man is better than any summer blockbuster we had to endure last summer , and hopefully , sets the tone for a summer of good stuff . if you're a comic fan , you can't miss it . if you're not , you'll still have a good time . "
0: the whole damn thing is ripe for the jerry springer crowd . it's all pretty cynical and condescending , too .
1: zhang yimou delivers warm , genuine characters who lie not through dishonesty , but because they genuinely believe it's the only way to bring happiness to their loved ones .
1: . . . a guiltless film for nice evening out .
0: is there enough material to merit a documentary on the making of wilco's last album ?
0: there's something deeply creepy about never again , a new arrow in schaeffer's quiver of ineptitudes .
0: really does feel like a short stretched out to feature length .
1: a live-wire film that never loses its ability to shock and amaze .
1: a compelling allegory about 

raccogliamo solo alcuni esempi (18) casualmente e notiamo che la label è 0 per i negativi e 1 per commenti positivi e di lunghezza variabile


---
### Exercise 1.2: A Pre-trained BERT and Tokenizer

The model we will use is a *very* small BERT transformer called [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) this model was trained (using self-supervised learning) on the same corpus as BERT but using the full BERT base model as a *teacher*.

**Your next task**: Load the DistilBERT model and corresponding tokenizer. Use the tokenizer on a few samples from the dataset and pass the tokens through the model to see what outputs are provided. I suggest you use the [`AutoModel`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html) class (and the `from_pretrained()` method) to load the model and `AutoTokenizer` to load the tokenizer).

In [6]:
# AutoClass imports.
from transformers import AutoTokenizer, AutoModel

model = AutoModel.from_pretrained('distilbert/distilbert-base-uncased')
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11463.61it/s]
DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
model

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [8]:
tokenizer

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

vediamo come funziona il tokenizer

In [9]:
tokens = tokenizer.encode("This is a test")
tokens

[101, 2023, 2003, 1037, 3231, 102]

In [10]:
tokens = tokenizer.encode("This is a test", return_tensors="pt")
tokens

tensor([[ 101, 2023, 2003, 1037, 3231,  102]])

vediamo cosa succede a fare il decode

In [11]:
tokenizer.decode(tokens)

['[CLS] this is a test [SEP]']

vediamo cosa restituisce il modello

In [12]:
output = model(tokens)
output

BaseModelOutput(last_hidden_state=tensor([[[-0.1565, -0.1862,  0.0528,  ..., -0.1188,  0.0662,  0.5470],
         [-0.3575, -0.6484, -0.0618,  ..., -0.3040,  0.3508,  0.5221],
         [-0.2772, -0.4459,  0.1818,  ..., -0.0948, -0.0076,  0.9958],
         [-0.2841, -0.3917,  0.3753,  ..., -0.2151, -0.1173,  1.0526],
         [ 0.2661, -0.5094, -0.3180,  ..., -0.4203,  0.0144, -0.2149],
         [ 0.9441,  0.0112, -0.4714,  ...,  0.1439, -0.7288, -0.1619]]],
       grad_fn=<NativeLayerNormBackward0>), hidden_states=None, attentions=None)

In [13]:
output.last_hidden_state.shape # 1 per il prompt, 6 per i token in ingresso, con anche CLS e SEP, tutti di dimensione

torch.Size([1, 6, 768])

se vogliamo il class token?

In [14]:
output.last_hidden_state[0][0].shape #class token

torch.Size([768])

Prendiamo i primi 5 prompt dal training dataset

In [15]:
batch = tokenizer(ds_train[:5]["text"], return_tensors="pt", padding=True) # abbiamo aggiunto il padding (token di pad)
batch

{'input_ids': tensor([[  101,  1996,  2600,  2003, 16036,  2000,  2022,  1996,  7398,  2301,
          1005,  1055,  2047,  1000, 16608,  1000,  1998,  2008,  2002,  1005,
          1055,  2183,  2000,  2191,  1037, 17624,  2130,  3618,  2084,  7779,
         29058,  8625, 13327,  1010,  3744,  1011, 18856, 19513,  3158,  5477,
          4168,  2030,  7112, 16562,  2140,  1012,   102,     0,     0,     0,
             0,     0],
        [  101,  1996,  9882,  2135,  9603, 13633,  1997,  1000,  1996,  2935,
          1997,  1996,  7635,  1000, 11544,  2003,  2061,  4121,  2008,  1037,
          5930,  1997,  2616,  3685, 23613,  6235,  2522,  1011,  3213,  1013,
          2472,  2848,  4027,  1005,  1055,  4423,  4432,  1997,  1046,  1012,
          1054,  1012,  1054,  1012, 23602,  1005,  1055,  2690,  1011,  3011,
          1012,   102],
        [  101,  4621,  2021,  2205,  1011,  8915, 23267, 16012, 24330,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     

che cosa è l'attention mask? Indica quali sono i token validi nell'attenzione

In [16]:
tokenizer.decode(batch["input_ids"]) # così stampiamo i prompt usati

['[CLS] the rock is destined to be the 21st century \' s new " conan " and that he \' s going to make a splash even greater than arnold schwarzenegger, jean - claud van damme or steven segal. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD]',
 '[CLS] the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co - writer / director peter jackson \' s expanded vision of j. r. r. tolkien \' s middle - earth. [SEP]',
 '[CLS] effective but too - tepid biopic [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]',
 '[CLS] if you sometimes like to go to the movies to have fun, wasabi is a good place to start. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [P

Vediamo l'importanza dell'attention mask

In [17]:
no_masking = model.forward(input_ids=batch["input_ids"]).last_hidden_state
no_masking

tensor([[[-0.0292,  0.0358,  0.0447,  ...,  0.0465,  0.5313,  0.2976],
         [-0.0091, -0.0533, -0.3162,  ...,  0.1485,  0.5437, -0.1091],
         [-0.0238, -0.0984, -0.1531,  ...,  0.1441,  0.2745, -0.1279],
         ...,
         [ 0.2498,  0.0141,  0.2388,  ...,  0.1595,  0.0821, -0.1611],
         [ 0.2471,  0.0118,  0.2464,  ...,  0.1922,  0.0874, -0.1682],
         [ 0.2161, -0.0090,  0.2404,  ...,  0.1839,  0.1098, -0.1339]],

        [[-0.2062, -0.0490, -0.4036,  ..., -0.1186,  0.6141,  0.3919],
         [-0.4361, -0.1647, -0.3533,  ...,  0.1086,  0.9478, -0.0272],
         [-0.1164,  0.1690,  0.2698,  ..., -0.1971,  0.4372,  0.2527],
         ...,
         [-0.2341,  0.4810, -0.2634,  ..., -0.3397,  0.2567,  0.1274],
         [ 0.7139,  0.0574, -0.3260,  ...,  0.2041, -0.3800, -0.3343],
         [ 0.5649,  0.2806, -0.0295,  ...,  0.1297, -0.3160, -0.1874]],

        [[ 0.0704, -0.3479,  0.2616,  ...,  0.0059,  0.4683,  0.0262],
         [-0.0984,  0.1426,  0.3205,  ..., -0

In [25]:
masking = model.forward(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state
masking

tensor([[[-3.3174e-02, -1.6809e-02,  1.9412e-02,  ...,  4.7614e-02,
           5.8341e-01,  3.0363e-01],
         [-2.3496e-02, -5.5522e-02, -3.6377e-01,  ...,  1.8773e-01,
           5.7809e-01, -1.5768e-01],
         [-5.1595e-02, -1.0141e-01, -1.5113e-01,  ...,  1.5027e-01,
           2.6485e-01, -1.5748e-01],
         ...,
         [ 3.6877e-01, -1.1471e-01,  8.4279e-01,  ..., -7.0794e-02,
          -1.7794e-02, -2.5157e-01],
         [ 6.5389e-02, -2.0571e-02,  1.8888e-01,  ...,  1.1587e-01,
           2.3229e-01, -2.4036e-01],
         [ 3.7301e-02, -1.0425e-02,  1.2027e-01,  ...,  1.0492e-01,
           2.8523e-01, -3.0345e-01]],

        [[-2.0616e-01, -4.8953e-02, -4.0360e-01,  ..., -1.1859e-01,
           6.1411e-01,  3.9191e-01],
         [-4.3609e-01, -1.6475e-01, -3.5332e-01,  ...,  1.0863e-01,
           9.4784e-01, -2.7189e-02],
         [-1.1638e-01,  1.6903e-01,  2.6975e-01,  ..., -1.9707e-01,
           4.3720e-01,  2.5274e-01],
         ...,
         [-2.3410e-01,  4

SHORTCUT

In [26]:
masking = model.forward(**batch).last_hidden_state
masking # otteniamo la stessa cosa

tensor([[[-3.3174e-02, -1.6809e-02,  1.9412e-02,  ...,  4.7614e-02,
           5.8341e-01,  3.0363e-01],
         [-2.3496e-02, -5.5522e-02, -3.6377e-01,  ...,  1.8773e-01,
           5.7809e-01, -1.5768e-01],
         [-5.1595e-02, -1.0141e-01, -1.5113e-01,  ...,  1.5027e-01,
           2.6485e-01, -1.5748e-01],
         ...,
         [ 3.6877e-01, -1.1471e-01,  8.4279e-01,  ..., -7.0794e-02,
          -1.7794e-02, -2.5157e-01],
         [ 6.5389e-02, -2.0571e-02,  1.8888e-01,  ...,  1.1587e-01,
           2.3229e-01, -2.4036e-01],
         [ 3.7301e-02, -1.0425e-02,  1.2027e-01,  ...,  1.0492e-01,
           2.8523e-01, -3.0345e-01]],

        [[-2.0616e-01, -4.8953e-02, -4.0360e-01,  ..., -1.1859e-01,
           6.1411e-01,  3.9191e-01],
         [-4.3609e-01, -1.6475e-01, -3.5332e-01,  ...,  1.0863e-01,
           9.4784e-01, -2.7189e-02],
         [-1.1638e-01,  1.6903e-01,  2.6975e-01,  ..., -1.9707e-01,
           4.3720e-01,  2.5274e-01],
         ...,
         [-2.3410e-01,  4


---
### Exercise 1.3: A Stable Baseline

In this exercise I want you to:
1. Use DistilBERT as a *feature extractor* to extract representations of the text strings from the dataset splits;
2. Train a classifier (your choice, by an SVM from Scikit-learn is an easy choice).
3. Evaluate performance on the validation and test splits.

These results are our *stable baseline* -- the **starting** point on which we will (hopefully) improve in the next exercise.

**Hint**: There are a number of ways to implement the feature extractor, but probably the best is to use a [feature extraction `pipeline`](https://huggingface.co/tasks/feature-extraction). You will need to interpret the output of the pipeline and extract only the `[CLS]` token from the *last* transformer layer. *How can you figure out which output that is?*

In [19]:
from transformers import pipeline
import torch
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# Your code here.

---
---
## Exercise 2: Fine-tuning DistilBERT

In this exercise we will fine-tune the DistilBERT model to (hopefully) improve sentiment analysis performance.


---
### Exercise 2.1: Token Preprocessing

The first thing we need to do is *tokenize* our dataset splits -- we don't want to re-tokenize our inputs for every batch! Our current datasets return a dictionary with *strings*, but we want *input token ids* (i.e. the output of the tokenizer). This is easy enough to do by hand, but the Hugging Face `Dataset` class provides convenient, efficient, and *lazy* methods. See the documentation for [`Dataset.map`](https://huggingface.co/docs/datasets/v3.5.0/en/package_reference/main_classes#datasets.Dataset.map).

**Tip**: Verify that your new datasets are returning for every element: `text`, `label`, `intput_ids`, and `attention_mask`.

In [20]:
# Your code here.


---
### Exercise 2.2: Setting up the Model to be Fine-tuned

In this exercise we need to prepare the base Distilbert model for fine-tuning for a *sequence classification task*. This means, at the very least, appending a new, randomly-initialized classification head connected to the `[CLS]` token of the last transformer layer. Luckily, HuggingFace already provides an `AutoModel` for just this type of instantiation: [`AutoModelForSequenceClassification`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html#automodelforsequenceclassification). You will want you instantiate one of these for fine-tuning.

In [21]:
from transformers import AutoModelForSequenceClassification

# Your code here.


---
### Exercise 2.3: Fine-tuning DistilBERT

Finally. In this exercise you should use a HuggingFace [`Trainer`](https://huggingface.co/docs/transformers/main/en/trainer) to fine-tune your model on the Rotten Tomatoes training split. Setting up the trainer will involve (at least):


1. Instantiating a [`DataCollatorWithPadding`](https://huggingface.co/docs/transformers/en/main_classes/data_collator) object which is what *actually* does your batch construction (by padding all sequences to the same length).
2. Writing an *evaluation function* that will measure the classification accuracy. This function takes a single argument which is a tuple containing `(logits, labels)` which you should use to compute classification accuracy (and maybe other metrics like F1 score, precision, recall) and return a `dict` with these metrics.  
3. Instantiating a [`TrainingArguments`](https://huggingface.co/docs/transformers/v4.51.1/en/main_classes/trainer#transformers.TrainingArguments) object using some reasonable defaults.
4. Instantiating a `Trainer` object using your train and validation splits, you data collator, and function to compute performance metrics.
5. Calling `trainer.train()`, waiting, waiting some more, and then calling `trainer.evaluate()` to see how it did.

**Tip**: When prototyping this laboratory I discovered the HuggingFace [Evaluate library](https://huggingface.co/docs/evaluate/en/index) which provides evaluation metrics. However I found it to have insufferable layers of abstraction and getting actual metrics computed. I suggest just using the Scikit-learn metrics...

In [22]:
# Your code here.


---
---
## Exercise 3: Choose your Own Adventure

As promised, you should choose **one** of the following exercises to work. Well, at *least* one. If you want to do them all, that is also OK! Or if you want to propose something else as a third exercise, reach out to me on the Discord!


---
### Exercise 3.1: Efficient Fine-tuning for Sentiment Analysis

In Exercise 2 we fine-tuned the *entire* Distilbert model on Rotten Tomatoes. This is expensive, even for a small model. Find an *efficient* way to fine-tune Distilbert on the Rotten Tomatoes dataset (or some other dataset).

**Hint**: You could check out the [HuggingFace PEFT library](https://huggingface.co/docs/peft/en/index) for some state-of-the-art approaches that should "just work". How else might you go about making fine-tuning more efficient without having to change your training pipeline from above?

**Why choose this exercise?** PEFT techniques -- especially LoRA are the methods of choice for adapting models to new tasks.

In [23]:
# Your code here.


---
### Exercise 3.2: Fine-tuning a CLIP Model (harder)

Use a (small) CLIP model like [`openai/clip-vit-base-patch16`](https://huggingface.co/openai/clip-vit-base-patch16) and evaluate its zero-shot performance on a small image classification dataset like ImageNette or TinyImageNet. Fine-tune (using a parameter-efficient method!) the CLIP model to see how much improvement you can squeeze out of it.

**Note**: There are several ways to adapt the CLIP model; you could fine-tune the image encoder, the text encoder, or both. Or, you could experiment with prompt learning.

**Tip**: CLIP probably already works very well on ImageNet and ImageNet-like images. For extra fun, look for an image classification dataset with different image types (e.g. *sketches*).

**Why choose this exercise?** CLIP is probably the most widely used Vision-Language Model, and adapting it is a useful skill to master.

In [24]:
# Your code here.


---
### Exercise 3.3: A Text-to-image Retrieval System (hard, but not *too* hard)

Implement a simple text-to-image retrieval system with a simple user interface --- using, for example, [gradio](https://www.gradio.app/), or [Marimo](https://marimo.io/), or [Shiny](https://shiny.posit.co/). Your application should *index* (e.g. compute visual descriptors for) a small dataset of images like [Flickr8k](https://huggingface.co/datasets/jxie/flickr8k). It should provide a user interface with which a user can enter a short text prompt (e.g. "a photo of dogs playing in the snow") and then display the top-10 matching images from the indexed dataset.

Note that there is no following code block with "Your code here" for this exercise. You will definitely want to implement this outside of a Jupyter Notebook.

**Hint**: The **CLIP** model is practically *made* for just such an application.

**Why choose this exercise?** Well, this is a course on Deep Learning *Applications*, and this is your chance to *build* one!

---
---